# Creating Generic Custom Lineage in Purview 

This notebook create a custom lineage in a Purview Account using the Atlas REST API. 
It runs in a Synapse session using Managed Identity.  
The Managed Identity is used to get secrets (TenantId, ClientId, ClientSecret) stored in a Key Vault.  
Those secrets will be used for the Purview Client Authentication.
Those secrets are associated with an existing service principal.
You can create this service principal using the Azure CLI command line below:

```bash
    az ad sp create-for-rbac --name "[ServicePName]" --role contributor --scopes /subscriptions/[subscriptionId]/resourceGroups/[ResourceGroupName] --sdk-auth
```
In order to allow the current notebook to create custom lineage in the Purview Account, you need to configure the Purview root collection to support this service principal as a Data Curators.  

To run this pipeline you need to specify:
- the name of the Microsoft Purview Account
- the name of the Azure Key Vault
- the name of the storage account where the dataset are stored
- the name of the container in the storage account where the dataset are stored
- the name of the secret where the tenant id of the service principal
- the name of the secret where the client id of the service principal
- the name of the secret where the client secret of the service principal
 


In [ ]:
%pip install pyapacheatlas==0.16.0 openlineage-python==1.37.0 azure-identity==1.16.1 azure-storage-blob==12.19.0
print("✓ Python packages installed successfully")

# Set Global Variables:

- the name of the Microsoft Purview Account
- the name of the Azure Key Vault
- the name of the storage account where the dataset are stored
- the name of the container in the storage account where the dataset are stored
- the name of the secret where the tenant id of the service principal
- the name of the secret where the client id of the service principal
- the name of the secret where the client secret of the service principal

In [ ]:
# Update the variables below before running this notebook

PURVIEW_ACCOUNT_NAME="to-be-completed"
STORAGE_ACCOUNT_NAME="to-be-completed"
CONTAINER_NAME="to-be-completed"
KEY_VAULT_NAME="to-be-completed"
SP_TENANT_ID_SECRET_NAME="to-be-completed"
SP_CLIENT_ID_SECRET_NAME="to-be-completed"
SP_SECRET_SECRET_NAME="to-be-completed"

print("✓ Python global variables set successfully")

In [ ]:
# Import required libraries
import json
from datetime import datetime, timedelta
from pyapacheatlas.auth import ServicePrincipalAuthentication
from pyapacheatlas.core import AtlasEntity, AtlasProcess
from pyapacheatlas.core.util import AtlasResponse
from azure.identity import DefaultAzureCredential
from pyapacheatlas.core import PurviewClient
import requests
import os
from notebookutils import mssparkutils

print("✓ Libraries imported successfully")

In [ ]:
# Retrieve the secrets from Key Vault using Managed Identity authentication
tenant_id=mssparkutils.credentials.getSecret(KEY_VAULT_NAME,SP_TENANT_ID_SECRET_NAME)
client_id=mssparkutils.credentials.getSecret(KEY_VAULT_NAME,SP_CLIENT_ID_SECRET_NAME)
client_secret=mssparkutils.credentials.getSecret(KEY_VAULT_NAME,SP_SECRET_SECRET_NAME)

# Authenticate using service principal
cred = ServicePrincipalAuthentication(
     tenant_id=tenant_id,
     client_id=client_id,
     client_secret=client_secret
)

# Create a Purview Client
client = PurviewClient(
    account_name=PURVIEW_ACCOUNT_NAME,
    authentication=cred
)
print("✓ Purview Client created successfully")

In [ ]:
# Define Source Table 1
ae_in01 = AtlasEntity(
    name=f"{CONTAINER_NAME}-pet.csv",
    typeName="azure_datalake_gen2_path",
    qualified_name=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{CONTAINER_NAME}/pet.csv",
    guid="-1",
)
print("Created PET Entity")

# Define Source Table 2
ae_in02 = AtlasEntity(
    name=f"{CONTAINER_NAME}-person.csv",
    typeName="azure_datalake_gen2_path",
    qualified_name=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{CONTAINER_NAME}/person.csv",
    guid="-2",
)
print("Created PERSON Entity")


# Define Output Table
ae_out = AtlasEntity(
    name=f"{CONTAINER_NAME}-all.csv",
    typeName="azure_datalake_gen2_path",
    qualified_name=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/CONTAINER_NAME/all.csv",
    guid="-3",
)
print("Created ALL Entity")


col_map = [
    {
        "DatasetMapping": {"Source": ae_in01.qualifiedName, "Sink": ae_out.qualifiedName},
        "ColumnMapping": [{"Source": "name", "animal": "age"}, {"Source": "name", "sex": "age"}],
    },
    {
        "DatasetMapping": {"Source": ae_in02.qualifiedName, "Sink": ae_out.qualifiedName},
        "ColumnMapping": [{"Source": "name", "sex": "age"}, {"Source": "name", "sex": "age"}],
    },
]

proc = AtlasProcess(
    name=f"{CONTAINER_NAME}-merge_pet_person",
    typeName="azure_synapse_operation",
    qualified_name="pet/person/all",
    guid="-4",
    inputs=[ae_in01, ae_in02],
    outputs=[ae_out],
)

print("Created Process")

proc.attributes.update(
    {
        "columnMapping": json.dumps(col_map),
    }
)


In [ ]:
results = client.upload_entities([proc, ae_in01, ae_in02, ae_out])
print("✓ Purview Custom Lineage created successfully")
print(f"Results: {json.dumps(results, indent=2)}")